### TensorFlow Graphs
<li>Nodes: တွက်ချက်မှုလုပ်ဆောင်ချက်များ (operations)</li>

<li>Edges: Tensor များ (ဒေတာများ)</li>

<p style="color:blue; font-weight:bold;">Graph Mode vs Eager Execution</p>

In [ ]:
import tensorflow as tf

# Eager mode - တွက်ချက်မှုကို ချက်ချင်းလုပ်ဆောင်သည်
a = tf.constant(5)
b = tf.constant(3)
c = a + b  # ချက်ချင်း တွက်ချက်သည်
print(c)   # tf.Tensor(8, shape=(), dtype=int32)

In [ ]:
import tensorflow as tf

# Graph mode - စုစည်းပြီး နောက်မှ တွက်ချက်သည်
@tf.function  # decorator ဖြင့် graph mode သို့ပြောင်း
def compute(a, b):
    return a + b

# Function ကို ပထမဆုံးအကြိမ် call လုပ်ချိန်မှ graph ကို create လုပ်သည်
result = compute(tf.constant(5), tf.constant(3))
print(result)  # tf.Tensor(8, shape=(), dtype=int32)

<p style="color:blue; font-weight:bold;">Graph တည်ဆောက်ပုံ နမူနာ</p>

In [ ]:
import tensorflow as tf

# Simple computation graph
def create_simple_graph():
    # Nodes/Operations
    a = tf.constant(2.0, name='a')
    b = tf.constant(3.0, name='b')
    c = tf.constant(4.0, name='c')
    
    # Operations
    d = tf.add(a, b, name='add')
    e = tf.multiply(d, c, name='multiply')
    
    return e

# Graph ကို run ရန်
result = create_simple_graph()
print("Result:", result.numpy())  # Result: 20.0

<p style="color:blue; font-weight:bold;">Graph Visualization with TensorBoard</p>

In [ ]:
import tensorflow as tf
import datetime

# Create a more complex graph
def create_graph_for_visualization():
    with tf.name_scope("Input_Layer"):
        x = tf.constant([[1.0, 2.0]], name='input')
    
    with tf.name_scope("Weights"):
        w = tf.Variable([[2.0], [3.0]], name='weights')
    
    with tf.name_scope("Operations"):
        b = tf.constant(1.0, name='bias')
        output = tf.matmul(x, w) + b
    
    return output

# Set up logging
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# Create and run graph
result = create_graph_for_visualization()

# TensorBoard ဖြင့် ကြည့်ရန် terminal တွင်:
# tensorboard --logdir logs/fit

<p style="color:blue; font-weight:bold;">Practical Example: Training with Graphs</p>

In [5]:
# Example: Linear Regression with Graphs (TensorFlow 2.x)
import tensorflow as tf
import numpy as np

# 1. Prepare dataset (y = 2x + 1 + noise)
X = np.linspace(-1, 1, 100).astype(np.float32)
y = (2 * X + 1 + np.random.randn(*X.shape) * 0.2).astype(np.float32)

# 2. Build a simple linear regression model
class LinearModel(tf.Module):
    def __init__(self):
        # trainable parameters
        self.W = tf.Variable(tf.random.normal([1]))
        self.b = tf.Variable(tf.random.normal([1]))

    @tf.function  # turn this function into a computation graph
    def __call__(self, x):
        return self.W * x + self.b

model = LinearModel()

# 3. Loss function (Mean Squared Error)
@tf.function
def loss_fn(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

# 4. Optimizer
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1)

# 5. Training step inside a graph
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        y_pred = model(x)
        loss = loss_fn(y, y_pred)
    gradients = tape.gradient(loss, [model.W, model.b])
    optimizer.apply_gradients(zip(gradients, [model.W, model.b]))
    return loss

# 6. Training loop
for epoch in range(100):
    loss = train_step(X, y)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss: {loss.numpy()}, W: {model.W.numpy()}, b: {model.b.numpy()}")   

# 7. Final parameters
print(f"Trained parameters: W: {model.W.numpy()}, b: {model.b.numpy()}")


Epoch 0: Loss: 0.5345320105552673, W: [1.078393], b: [0.6886164]
Epoch 10: Loss: 0.11375472694635391, W: [1.5429642], b: [0.9856391]
Epoch 20: Loss: 0.05072733759880066, W: [1.7726578], b: [1.0175316]
Epoch 30: Loss: 0.0357794314622879, W: [1.886223], b: [1.020956]
Epoch 40: Loss: 0.03213068097829819, W: [1.9423718], b: [1.0213238]
Epoch 50: Loss: 0.031238801777362823, W: [1.9701331], b: [1.0213633]
Epoch 60: Loss: 0.031020784750580788, W: [1.9838588], b: [1.0213674]
Epoch 70: Loss: 0.030967488884925842, W: [1.9906452], b: [1.0213678]
Epoch 80: Loss: 0.03095446154475212, W: [1.9940004], b: [1.0213678]
Epoch 90: Loss: 0.030951276421546936, W: [1.9956592], b: [1.0213678]
Trained parameters: W: [1.996421], b: [1.0213678]
